In [1]:
# Import all necessary libraries for the project
import os
import shutil
from pathlib import Path
from collections import defaultdict
import hashlib
from PIL import Image
import imagehash
from datetime import datetime
import json
import csv
import torch
import clip
import numpy as np
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
import pickle
from tqdm import tqdm
import base64
import io

In [4]:
# Create all necessary directories for the project
output_dir = Path("photo_clustering_project")
raw_images_dir = output_dir / "raw_images"
deduplicated_dir = output_dir / "deduplicated_images"
sample_dir = output_dir / "sample_dataset"
reports_dir = output_dir / "reports"
features_dir = output_dir / "features"
clusters_dir = output_dir / "clusters"
export_dir = output_dir / "nextjs_export"
images_export_dir = export_dir / "images"

for directory in [raw_images_dir, deduplicated_dir, sample_dir, reports_dir, 
                    features_dir, clusters_dir, export_dir, images_export_dir]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"✓ Created project structure at: {output_dir.absolute()}")

✓ Created project structure at: d:\Projects\IMAGE-CLUSTERING\photo_clustering_project


In [5]:
# Define your image source folders with labels
source_configs = [
    {
        'path': 'F:\\Dev\\Images',  # Change this to your personal folder path
        'label': 'personal'
    },
    {
        'path': 'F:\\General\\Mumy\\Images',  # Change this to your mother's folder path
        'label': 'mother'
    }
]

image_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp', '.tiff', '.heic'}
image_metadata = []

print("✓ Source folders configured")
print(f"  Personal: {source_configs[0]['path']}")
print(f"  Mother: {source_configs[1]['path']}")

✓ Source folders configured
  Personal: F:\Dev\Images
  Mother: F:\General\Mumy\Images


In [6]:
# Collect all images from source folders with metadata tracking
collected = 0

for config in source_configs:
    source = Path(config['path'])
    label = config['label']
    
    if not source.exists():
        print(f"⚠ Warning: {source} does not exist, skipping...")
        continue
    
    print(f"\n📁 Processing: {label} folder")
    folder_count = 0
    
    for file_path in source.rglob('*'):
        if file_path.suffix.lower() in image_extensions:
            try:
                dest_name = f"{label}_{collected:06d}_{file_path.name}"
                dest_path = raw_images_dir / dest_name
                shutil.copy2(file_path, dest_path)
                
                image_metadata.append({
                    'new_path': str(dest_path),
                    'original_path': str(file_path),
                    'source_label': label,
                    'filename': file_path.name,
                    'size_bytes': file_path.stat().st_size,
                    'format': file_path.suffix.lower(),
                    'collected_id': collected
                })
                
                collected += 1
                folder_count += 1
                
                if collected % 100 == 0:
                    print(f"  Collected {collected} images...", end='\r')
                    
            except Exception as e:
                print(f"\n⚠ Error: {e}")
    
    print(f"  ✓ {folder_count} images from {label}")

print(f"\n✅ Total collected: {collected} images")


📁 Processing: personal folder
  ✓ 230 images from personal

📁 Processing: mother folder
  ✓ 6376 images from mother

✅ Total collected: 6606 images


In [7]:
# Scan for duplicate images using perceptual hashing
print("\n🔍 Scanning for duplicates...")

hash_groups = defaultdict(list)
exact_duplicates = defaultdict(list)
images = list(raw_images_dir.glob('*'))

for idx, img_path in enumerate(images):
    if img_path.suffix.lower() not in image_extensions:
        continue
    
    try:
        img = Image.open(img_path)
        phash = str(imagehash.phash(img))
        
        with open(img_path, 'rb') as f:
            file_hash = hashlib.md5(f.read()).hexdigest()
        
        width, height = img.size
        hash_groups[phash].append(img_path)
        exact_duplicates[file_hash].append(img_path)
        
        for meta in image_metadata:
            if Path(meta['new_path']) == img_path:
                meta['width'] = width
                meta['height'] = height
                meta['aspect_ratio'] = width/height if height > 0 else 0
        
        if (idx + 1) % 100 == 0:
            print(f"  Processed {idx + 1}/{len(images)}...", end='\r')
            
    except Exception as e:
        pass

print(f"\n✓ Processed {len(images)} images")

# Find similar groups
duplicate_groups = []
all_hashes = list(hash_groups.keys())

for i, hash1 in enumerate(all_hashes):
    group = set(hash_groups[hash1])
    for hash2 in all_hashes[i+1:]:
        diff = imagehash.hex_to_hash(hash1) - imagehash.hex_to_hash(hash2)
        if diff <= 5:
            group.update(hash_groups[hash2])
    if len(group) > 1:
        duplicate_groups.append(list(group))

exact_dups = sum(1 for files in exact_duplicates.values() if len(files) > 1)
print(f"✓ Found {exact_dups} exact duplicates")
print(f"✓ Found {len(duplicate_groups)} similar groups")


🔍 Scanning for duplicates...


d:\Language-Setup\PYTHON\Lib\site-packages\PIL\Image.py:3432: DecompressionBombWarning: Image size (108576768 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(



✓ Processed 6606 images
✓ Found 26 exact duplicates
✓ Found 553 similar groups


In [8]:
# Remove duplicate images keeping the largest version
print("\n🗑️  Removing duplicates...")

all_images = set(raw_images_dir.glob('*'))
duplicate_images = set()
for group in duplicate_groups:
    duplicate_images.update(group)

unique_images = all_images - duplicate_images
kept_images = set()
removed_count = 0

for img_path in unique_images:
    if img_path.suffix.lower() in image_extensions:
        shutil.copy2(img_path, deduplicated_dir / img_path.name)
        kept_images.add(img_path)

for group in duplicate_groups:
    keep = max(group, key=lambda x: x.stat().st_size)
    shutil.copy2(keep, deduplicated_dir / keep.name)
    kept_images.add(keep)
    removed_count += len(group) - 1

image_metadata = [meta for meta in image_metadata if Path(meta['new_path']) in kept_images]

for meta in image_metadata:
    old_path = Path(meta['new_path'])
    meta['new_path'] = str(deduplicated_dir / old_path.name)

print(f"✓ Kept {len(kept_images)} unique images")
print(f"✓ Removed {removed_count} duplicates")


🗑️  Removing duplicates...
✓ Kept 5984 unique images
✓ Removed 681 duplicates


In [9]:
# Create a sample dataset using ALL images
all_images = [img for img in deduplicated_dir.glob('*') if img.suffix.lower() in image_extensions]

sample = all_images  # no balancing, no sampling, no proportions

for img_path in sample:
    shutil.copy2(img_path, sample_dir / img_path.name)

print(f"✅ Created sample: {len(sample)} images")

✅ Created sample: 5984 images


In [10]:
# Load CLIP model for feature extraction
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {device}")

model, preprocess = clip.load("ViT-B/32", device=device)
print("✅ CLIP model loaded")

🖥️  Device: cuda
✅ CLIP model loaded


In [11]:
# Extract CLIP features from all sample images
print("\n🎨 Extracting features...")

image_files = sorted([f for f in sample_dir.glob('*') if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp'}])

features_list = []
valid_images = []

for img_path in tqdm(image_files, desc="Processing"):
    try:
        image = preprocess(Image.open(img_path)).unsqueeze(0).to(device)
        with torch.no_grad():
            features = model.encode_image(image)
            features = features.cpu().numpy().flatten()
        features_list.append(features)
        valid_images.append(str(img_path))
    except Exception as e:
        print(f"⚠ Error: {img_path.name}")

features_array = np.array(features_list)
features_array = normalize(features_array, axis=1)

np.save(features_dir / "clip_features.npy", features_array)
with open(features_dir / "image_list.json", 'w') as f:
    json.dump(valid_images, f, indent=2)

print(f"\n✅ Extracted {len(valid_images)} features")
print(f"   Shape: {features_array.shape}")


🎨 Extracting features...


Processing: 100%|██████████| 5974/5974 [17:22<00:00,  5.73it/s]



✅ Extracted 5974 features
   Shape: (5974, 512)


In [12]:
# Define category descriptions for classification
level1_categories = {
    'personal_solo': 'a selfie or solo portrait photo of a person',
    'personal_friends': 'a photo of people with friends in casual settings',
    'family_relatives': 'a family photo with relatives or group family gathering',
    'relatives_formal': 'a formal photo with relatives at an event',
    'journey_travel': 'travel photos, tourist locations, vacation pictures',
    'journey_outdoor': 'outdoor activity photos, hiking, adventure',
    'nature_landscape': 'nature landscape, mountains, scenery, outdoor views',
    'nature_closeup': 'close-up nature photo, flowers, plants',
    'jewelry_accessories': 'jewelry, gold ornaments, accessories, rings, necklaces',
    'clothing_fashion': 'clothing items, fashion, dresses, traditional wear',
    'religious_gods': 'religious images, deity photos, temple pictures, spiritual',
    'screenshots': 'phone screenshots, app screenshots, text messages',
    'receipts_bills': 'bills, receipts, invoices, documents with text',
    'food': 'food photos, meals, dishes, cooking',
    'events_celebrations': 'birthday parties, weddings, celebrations, festivals',
    'pets_animals': 'photos of pets, dogs, cats, animals',
    'indoor_home': 'indoor photos, home interior, room photos',
    'other': 'miscellaneous photos'
}

print(f"✅ Defined {len(level1_categories)} categories")

✅ Defined 18 categories


In [13]:
# Classify images into main categories using CLIP
print("\n🏷️  Classifying images...")

text_tokens = clip.tokenize(list(level1_categories.values())).to(device)
with torch.no_grad():
    text_features = model.encode_text(text_tokens).cpu().numpy()
text_features = normalize(text_features, axis=1)

similarity = features_array @ text_features.T
category_indices = np.argmax(similarity, axis=1)
confidence_scores = np.max(similarity, axis=1)
category_names = list(level1_categories.keys())

assignments = []
category_counts = {}

for idx, (cat_idx, confidence) in enumerate(zip(category_indices, confidence_scores)):
    category = category_names[cat_idx]
    assignments.append({
        'image_idx': idx,
        'category': category,
        'confidence': float(confidence)
    })
    category_counts[category] = category_counts.get(category, 0) + 1

print("\n📊 Category Distribution:")
print("-" * 50)
for category, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    percentage = (count / len(assignments)) * 100
    print(f"  {category:25s}: {count:4d} ({percentage:5.1f}%)")


🏷️  Classifying images...

📊 Category Distribution:
--------------------------------------------------
  clothing_fashion         : 1642 ( 27.5%)
  relatives_formal         : 1005 ( 16.8%)
  religious_gods           :  713 ( 11.9%)
  other                    :  549 (  9.2%)
  jewelry_accessories      :  410 (  6.9%)
  screenshots              :  308 (  5.2%)
  receipts_bills           :  285 (  4.8%)
  events_celebrations      :  271 (  4.5%)
  personal_solo            :  262 (  4.4%)
  personal_friends         :  178 (  3.0%)
  journey_travel           :  109 (  1.8%)
  journey_outdoor          :   74 (  1.2%)
  family_relatives         :   58 (  1.0%)
  nature_landscape         :   44 (  0.7%)
  food                     :   43 (  0.7%)
  indoor_home              :   15 (  0.3%)
  pets_animals             :    6 (  0.1%)
  nature_closeup           :    2 (  0.0%)


In [14]:
# Create sub-clusters within each category
print("\n🌳 Creating sub-clusters...")

n_subclusters = 3
hierarchical_clusters = {}

for category in set(a['category'] for a in assignments):
    indices = [a['image_idx'] for a in assignments if a['category'] == category]
    
    if len(indices) < n_subclusters:
        hierarchical_clusters[category] = {
            'subclusters': [{'indices': indices, 'subcluster_id': 0, 'size': len(indices)}],
            'n_subclusters': 1
        }
        continue
    
    category_features = features_array[indices]
    kmeans = KMeans(n_clusters=min(n_subclusters, len(indices)), random_state=42)
    subcluster_labels = kmeans.fit_predict(category_features)
    
    subclusters = []
    for sub_id in range(kmeans.n_clusters):
        sub_indices = [indices[i] for i, label in enumerate(subcluster_labels) if label == sub_id]
        subclusters.append({'subcluster_id': sub_id, 'indices': sub_indices, 'size': len(sub_indices)})
    
    hierarchical_clusters[category] = {
        'subclusters': subclusters,
        'n_subclusters': len(subclusters)
    }
    
    print(f"  {category}: {len(indices)} → {len(subclusters)} sub-clusters")


🌳 Creating sub-clusters...
  religious_gods: 713 → 3 sub-clusters
  other: 549 → 3 sub-clusters
  journey_travel: 109 → 3 sub-clusters
  journey_outdoor: 74 → 3 sub-clusters
  personal_friends: 178 → 3 sub-clusters
  events_celebrations: 271 → 3 sub-clusters
  family_relatives: 58 → 3 sub-clusters
  personal_solo: 262 → 3 sub-clusters
  indoor_home: 15 → 3 sub-clusters
  relatives_formal: 1005 → 3 sub-clusters
  receipts_bills: 285 → 3 sub-clusters
  pets_animals: 6 → 3 sub-clusters
  jewelry_accessories: 410 → 3 sub-clusters
  screenshots: 308 → 3 sub-clusters
  nature_landscape: 44 → 3 sub-clusters
  clothing_fashion: 1642 → 3 sub-clusters
  food: 43 → 3 sub-clusters


In [15]:
# Save complete clustering results
results = {
    'timestamp': datetime.now().isoformat(),
    'total_images': len(valid_images),
    'level1_categories': list(level1_categories.keys()),
    'images': []
}

for idx, img_path in enumerate(valid_images):
    level1_info = next(a for a in assignments if a['image_idx'] == idx)
    category = level1_info['category']
    
    subcluster_id = None
    for subcluster in hierarchical_clusters[category]['subclusters']:
        if idx in subcluster['indices']:
            subcluster_id = subcluster['subcluster_id']
            break
    
    results['images'].append({
        'index': idx,
        'path': img_path,
        'filename': Path(img_path).name,
        'level1_category': category,
        'level1_confidence': level1_info['confidence'],
        'level2_subcluster': subcluster_id,
        'cluster_full_id': f"{category}_{subcluster_id}"
    })

with open(clusters_dir / "clustering_results.json", 'w') as f:
    json.dump(results, f, indent=2)

with open(clusters_dir / "clustering_results.pkl", 'wb') as f:
    pickle.dump({'results': results, 'hierarchical_clusters': hierarchical_clusters, 'level1_assignments': assignments}, f)

print("✅ Clustering results saved")

✅ Clustering results saved


In [16]:
# Create 3D embeddings using PCA for visualization
print("\n🎨 Creating 3D embeddings...")

pca = PCA(n_components=3, random_state=42)
embeddings_3d = pca.fit_transform(features_array)

for i in range(3):
    min_val = embeddings_3d[:, i].min()
    max_val = embeddings_3d[:, i].max()
    embeddings_3d[:, i] = (embeddings_3d[:, i] - min_val) / (max_val - min_val)

embeddings_3d = (embeddings_3d - 0.5) * 200

print(f"✅ Created 3D embeddings: {embeddings_3d.shape}")
print(f"   Explained variance: {pca.explained_variance_ratio_.sum():.2%}")


🎨 Creating 3D embeddings...
✅ Created 3D embeddings: (5974, 3)
   Explained variance: 26.01%


In [17]:
# Compute k-nearest neighbors for similarity connections
print("\n🔍 Computing similarities...")

n_neighbors = 10
nn_model = NearestNeighbors(n_neighbors=n_neighbors + 1, metric='cosine')
nn_model.fit(features_array)

distances, indices = nn_model.kneighbors(features_array)

similarities = []
for i in range(len(features_array)):
    similar = []
    for j in range(1, len(indices[i])):
        similar.append({
            'index': int(indices[i][j]),
            'similarity': float(1 - distances[i][j])
        })
    similarities.append(similar)

print(f"✅ Computed similarities for {len(similarities)} images")


🔍 Computing similarities...
✅ Computed similarities for 5974 images


In [18]:
# Copy images to export folder for Next.js
print("\n📸 Exporting images...")

exported_images = []

for idx, img_path_str in enumerate(valid_images):
    img_path = Path(img_path_str)
    
    if not img_path.exists():
        continue
    
    try:
        new_filename = f"img_{idx:04d}{img_path.suffix}"
        dest_path = images_export_dir / new_filename
        shutil.copy2(img_path, dest_path)
        
        with Image.open(img_path) as img:
            img.thumbnail((200, 200), Image.Resampling.LANCZOS)
            thumb_path = images_export_dir / f"thumb_{idx:04d}.jpg"
            img.save(thumb_path, 'JPEG', quality=85)
        
        exported_images.append({
            'index': idx,
            'filename': new_filename,
            'thumbnail': f"thumb_{idx:04d}.jpg",
            'original_name': img_path.name
        })
        
        if (idx + 1) % 100 == 0:
            print(f"  Processed {idx + 1}/{len(valid_images)}...", end='\r')
            
    except Exception as e:
        pass

print(f"\n✅ Exported {len(exported_images)} images")


📸 Exporting images...


d:\Language-Setup\PYTHON\Lib\site-packages\PIL\Image.py:3432: DecompressionBombWarning: Image size (108576768 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(



✅ Exported 5971 images


In [19]:
# Create final visualization data structure
category_colors = {
    'personal_solo': '#FF6B6B', 'personal_friends': '#4ECDC4',
    'family_relatives': '#45B7D1', 'relatives_formal': '#96CEB4',
    'journey_travel': '#FFEAA7', 'journey_outdoor': '#DFE6E9',
    'nature_landscape': '#74B9FF', 'nature_closeup': '#55EFC4',
    'jewelry_accessories': '#FDCB6E', 'clothing_fashion': '#FD79A8',
    'religious_gods': '#A29BFE', 'screenshots': '#636E72',
    'receipts_bills': '#B2BEC3', 'food': '#FF7675',
    'events_celebrations': '#FD79A8', 'pets_animals': '#00B894',
    'indoor_home': '#6C5CE7', 'other': '#95A5A6'
}

viz_data = {
    'metadata': {
        'total_images': len(valid_images),
        'total_categories': len(set(img['level1_category'] for img in results['images'])),
        'export_date': datetime.now().isoformat(),
        'embeddings_method': 'PCA'
    },
    'categories': category_colors,
    'images': []
}

for idx, img_info in enumerate(results['images']):
    viz_data['images'].append({
        'id': idx,
        'filename': img_info['filename'],
        'position': {
            'x': float(embeddings_3d[idx, 0]),
            'y': float(embeddings_3d[idx, 1]),
            'z': float(embeddings_3d[idx, 2])
        },
        'category': img_info['level1_category'],
        'subcluster': img_info['level2_subcluster'],
        'confidence': float(img_info['level1_confidence']),
        'color': category_colors.get(img_info['level1_category'], '#95A5A6'),
        'similar': similarities[idx]
    })

print(f"✅ Created visualization data for {len(viz_data['images'])} images")

✅ Created visualization data for 5974 images


In [ ]:
# Save all JSON files for Next.js application
stats = {}
for category in category_colors.keys():
    images = [img for img in results['images'] if img['level1_category'] == category]
    if images:
        stats[category] = {
            'count': len(images),
            'color': category_colors[category],
            'subclusters': {}
        }

with open(export_dir / "visualization_data.json", 'w') as f:
    json.dump(viz_data, f, indent=2)

with open(export_dir / "category_stats.json", 'w') as f:
    json.dump(stats, f, indent=2)

with open(export_dir / "image_mappings.json", 'w') as f:
    json.dump(exported_images, f, indent=2)

print("JSON Files are saved for application")

✅ ALL PHASES COMPLETE!

📁 Export location: d:\Projects\IMAGE-CLUSTERING\photo_clustering_project\nextjs_export

📊 Summary:
  • Total images: 5974
  • Categories: 18
  • 3D embeddings: ✓
  • Similarities: ✓

🚀 Next: Copy 'nextjs_export' to your Next.js project!
